# **Build knowledge graph**

Builds a graph of legal entities and relations from the parent chunks already indexed by `deploy.ipynb`.
Every node and edge keeps the ids of the parent chunks it came from, so graph retrieval plugs into the existing rerank + MongoDB lookup.

Steps:
1. **Extract**: one LLM call per parent chunk, cached in MongoDB `graph_extractions`
2. **Merge**: fold cached extractions into `graph_entities` / `graph_relations` (no LLM calls)
3. **Index**: embed entity / relation descriptions into Qdrant `<COLLECTION_NAME>_entities` / `_relations`

## **Setup**

In [1]:
import os
import sys
from pathlib import Path

os.environ["GRPC_VERBOSITY"] = "ERROR"

from dotenv import load_dotenv

BACKEND_DIR = Path("../App/backend/src").resolve()
load_dotenv(BACKEND_DIR / ".env")
sys.path.insert(0, str(BACKEND_DIR))

from search_document_v2 import graph_builder as G

print("LLM:", os.getenv("GEMINI_MODEL"), "| Qdrant:", G.ENTITY_COLLECTION, "/", G.RELATION_COLLECTION)

/Users/duongvu/Projects/law_Q_A_system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: gemini-3.5-flash-lite | Qdrant: law_collection_entities / law_collection_relations


## **1. Extract**

In [3]:
TITLE = None                   # None = whole corpus
LIMIT = None                   # max parents this run; None = all pending
REQUESTS_PER_MINUTE = 12
MAX_REQUESTS = None
THINKING_BUDGET = 1024            

try:
    G.extract_graph(
        title=TITLE,
        limit=LIMIT,
        requests_per_minute=REQUESTS_PER_MINUTE,
        max_requests=MAX_REQUESTS,
        thinking_budget=THINKING_BUDGET,
    )
except G.DailyQuotaExceeded as e:
    print(e, "- rerun this cell after the daily quota resets")

0 parents to extract (2880 already cached)


In [5]:
db = G._graph_db()

In [ ]:
print("cached extractions:", db["graph_extractions"].count_documents({}))
for ext in db["graph_extractions"].find().limit(2):
    print(f"\n=== {ext['_id']} ({ext.get('model')}) ===")
    for e in ext["entities"][:8]:
        print(f"  [{e['type']}] {e['name']}: {e['description'][:120]}")
    for r in ext["relations"][:5]:
        print(f"  {r['source']} --{r.get('keywords', '')}--> {r['target']}")

cached extractions: 91

=== 142881_0 (gemini-3.6-flash) ===
  [văn bản pháp luật] Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam: Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam là cơ sở pháp lý để Quốc hội ban hành Luật Doanh nghiệp.
  [cơ quan nhà nước] Quốc hội: Quốc hội là cơ quan ban hành Luật Doanh nghiệp dựa trên căn cứ Hiến pháp.
  [văn bản pháp luật] Luật Doanh nghiệp: Luật Doanh nghiệp là văn bản quy phạm pháp luật do Quốc hội ban hành.
  Quốc hội --['ban hành']--> Luật Doanh nghiệp
  Luật Doanh nghiệp --['căn cứ']--> Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam

=== 142881_1 (gemini-3.6-flash) ===
  [loại hình doanh nghiệp] doanh nghiệp: Là đối tượng áp dụng và điều chỉnh của Luật Doanh nghiệp, bao gồm công ty trách nhiệm hữu hạn, công ty cổ phần, công ty 
  [loại hình doanh nghiệp] công ty trách nhiệm hữu hạn: Làm một trong các loại hình doanh nghiệp thuộc phạm vi điều chỉnh của Luật Doanh nghiệp theo Điều 1.
  [loại hình doanh nghiệp] công ty cổ phần: Là một loại

## **2. Merge**
Rebuilds the merged graph from all cached extractions. No LLM calls; rerun after every extraction run.

In [6]:
G.merge_graph()

top = db["graph_entities"].find().sort("degree", -1).limit(10)
for e in top:
    print(f"{e['degree']:>4} relations | {len(e['parent_ids']):>4} parents | [{e['type']}] {e['name']}")

merged: 4715 entities, 6504 relations
 438 relations |  462 parents | [chủ thể] doanh nghiệp
 413 relations |  714 parents | [cơ quan nhà nước] Bộ Tài chính
 244 relations |  297 parents | [loại hình doanh nghiệp] doanh nghiệp nhà nước
 240 relations |  418 parents | [cơ quan nhà nước] Thủ tướng Chính phủ
 132 relations |  224 parents | [cơ quan nhà nước] Bộ Kế hoạch và Đầu tư
 118 relations |  247 parents | [loại hình doanh nghiệp] công ty cổ phần
  96 relations |  152 parents | [cơ quan nhà nước] Chính phủ
  95 relations |  147 parents | [cơ quan nhà nước] cơ quan đại diện chủ sở hữu
  84 relations |  145 parents | [bộ máy quản lý] Hội đồng thành viên
  83 relations |  114 parents | [chủ thể] người lao động


## **3. Index**
Embeds every entity and relation description (one embedding call per 8 descriptions). Point ids derive from the node / edge keys, so rerunning overwrites; if interrupted, rerun with the `start` printed in the error.

In [7]:
G.index_graph(start=0)

entity: 16 / 4715
entity: 32 / 4715
entity: 48 / 4715
entity: 64 / 4715
entity: 80 / 4715
entity: 96 / 4715
entity: 112 / 4715
entity: 128 / 4715
entity: 144 / 4715
entity: 160 / 4715
entity: 176 / 4715
entity: 192 / 4715
entity: 208 / 4715
entity: 224 / 4715
entity: 240 / 4715
entity: 256 / 4715
entity: 272 / 4715
entity: 288 / 4715
entity: 304 / 4715
entity: 320 / 4715
entity: 336 / 4715
entity: 352 / 4715
entity: 368 / 4715
entity: 384 / 4715
entity: 400 / 4715
entity: 416 / 4715
entity: 432 / 4715
entity: 448 / 4715
entity: 464 / 4715
entity: 480 / 4715
entity: 496 / 4715
entity: 512 / 4715
entity: 528 / 4715
entity: 544 / 4715
entity: 560 / 4715
entity: 576 / 4715
entity: 592 / 4715
entity: 608 / 4715
entity: 624 / 4715
entity: 640 / 4715
entity: 656 / 4715
entity: 672 / 4715
entity: 688 / 4715
entity: 704 / 4715
entity: 720 / 4715
entity: 736 / 4715
entity: 752 / 4715
entity: 768 / 4715
entity: 784 / 4715
entity: 800 / 4715
entity: 816 / 4715
entity: 832 / 4715
entity: 848 / 4715

(4715, 6504)

In [8]:
from qdrant_client import QdrantClient
from search_document_v2.vector_store import QDRANT_URL, QDRANT_API_KEY

q = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
print("Mongo entities:", db["graph_entities"].count_documents({}), "| Qdrant:", q.count(G.ENTITY_COLLECTION).count)
print("Mongo relations:", db["graph_relations"].count_documents({}), "| Qdrant:", q.count(G.RELATION_COLLECTION).count)

Mongo entities: 4715 | Qdrant: 4715
Mongo relations: 6504 | Qdrant: 6504
